# 04 — Programs made of gates: blocks, backwards, and borrowed qubits

## What you will learn

Every circuit so far has been written one gate at a time, by hand. That stops working
almost immediately: the algorithms in the second half of this project are hundreds of
gates long, and they are not written as hundreds of gates — they are written as a
handful of *named pieces*, each of which is used, inverted, and conditioned on other
qubits as if it were a single gate.

This notebook builds those pieces. By the end you will know:

- how `@qsim.gate` turns a function of qubits into a **block** — a named, reusable
  chunk of circuit — and why a circuit then has two different sizes: what it is *made
  of* (`block_counts()`) and what it *costs* (`gate_counts()`);
- what it means to run a program **backwards**, why every quantum program can be run
  backwards, and the precise sense in which a classical `AND` gate cannot;
- the one operation that has no reverse — **measurement** — and what qsim does when you
  try to put one inside a scope that would have to undo it;
- how to run a whole block **only if** another qubit says so, while that qubit is itself
  in superposition, producing a superposition *of the program having run and not having
  run*;
- **ancillas**: scratch qubits you borrow and must give back clean, and the trap that
  makes "clean" a physical requirement rather than tidiness — a scratch qubit that still
  remembers what you did to it silently destroys the interference your algorithm was
  relying on, *even though nobody ever looked at it*;
- **Bennett's trick** — compute, copy out the answer, uncompute the scratch — which is
  the discipline that all of the reversible arithmetic in Phase 4 is built on.

The heading of this notebook says "software engineering", and the first half of it is.
Then the physics walks in and takes over, in section 4, and never leaves.

Here is the shape of the problem. Notebook 03's teleportation circuit was already at the
edge of what is comfortable to write out gate by gate, and it is under ten operations
long. Shor's algorithm on a 15-bit number is tens of thousands. Nobody writes those by
hand, and nobody would understand them if they did.

Classical programming solved this with the subroutine, and the answer here is the same
word — but a quantum subroutine has to support two operations an ordinary Python
function does not:

1. **Invert it.** Run the whole block backwards, undoing what it did.
2. **Control it.** Run the whole block only in the branches where some other qubit is
   $|1\rangle$.

Both of these have to work on the block *as a unit*, without you writing a second,
hand-inverted version of it. And both of them turn out to be possible for exactly one
reason: a block of gates is a **unitary**, and unitaries can always be inverted and
always be conditioned. That is the thread running through this whole notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import qsim
from qsim import Circuit, Qubit, Register, viz
from qsim.errors import DirtyAncillaError, NoCloningError, QsimError
from qsim.gates import CNOT, H, Ry, Rz, S, T, Toffoli, X

np.set_printoptions(precision=3, suppress=True)

## 1. `@qsim.gate` — giving a piece of circuit a name

A **block** is a Python function that takes qubit handles and applies gates to them,
decorated with `@qsim.gate`. That is the whole idea. The decorator does not change what
the function does when you call it; it changes what the *library* can do with it
afterwards.

In [ ]:
@qsim.gate
def bell(a: Qubit, b: Qubit) -> None:
    """Turn two fresh qubits into the Bell pair (|00⟩ + |11⟩)/√2."""
    H(a)
    CNOT(a, b)


qc = Circuit(name="blocks", seed=0)
a, b = qc.alloc_many(2)
bell(a, b)
print(qc.inspect.ket())

Nothing surprising: calling `bell(a, b)` ran `H` and then `CNOT`, immediately, exactly as
if you had typed them. Execution in qsim is **eager** — a gate applied is a gate run —
and a block does not change that.

What the block *does* change is the record. A circuit can now answer two different
questions about itself, and they are genuinely different questions:

- **What is this circuit made of?** — `block_counts()`. This is how you wrote it.
- **What does this circuit cost?** — `gate_counts()`. This is what actually ran on the
  qubits, in elementary one- and two-qubit gates, which is the only currency real
  hardware trades in.

In [ ]:
print("block_counts():", qc.block_counts())
print("gate_counts(): ", qc.gate_counts())
print()
for op in qc.history:
    print(f"  {op.name:<5} targets={op.qubit_ids}  controls={op.controls}  block={op.block!r}")

Read the history carefully, because two things in it matter later.

First, **there is no `bell` in the history.** The history is a list of elementary
operations, and each one carries a `block` *tag* saying which block it came from. The
block is a label on the ops, not a container holding them. That is deliberate: the state
tensor only ever sees ordinary gates, so a block cannot smuggle in anything that is not
a gate.

Second, look at how `CNOT` is stored: `targets=(1,)`, `controls=(0,)`. A controlled gate
is recorded as *the gate plus a list of control qubits*. On paper, a gate on two qubits
is a $4\times4$ matrix and $\mathrm{CNOT}$ is a perfectly good one — but qsim never
builds it. It applies the $2\times2$ matrix $X$ to the slice of the state where qubit 0
reads $1$, and leaves the other slice alone. Hold on to that; section 3 is entirely
about it.

Blocks nest, since a block's body is just Python and may call other blocks:

In [ ]:
@qsim.gate
def ghz(r: Register) -> None:
    """Extend a Bell pair to the three-qubit GHZ state (|000⟩ + |111⟩)/√2."""
    bell(r[0], r[1])
    CNOT(r[1], r[2])


qc = Circuit(name="ghz", seed=0)
r = qc.register(3, name="q")
ghz(r)
print(qc.inspect.ket())
print()
print("block_counts():", qc.block_counts())
print("gate_counts(): ", qc.gate_counts())

Both blocks are counted — one `ghz` call, and the one `bell` call it made inside — while
the gate count stays what it always was: three elementary gates, `H` and two `CNOT`s.
The two numbers answer the two questions, and neither is a substitute for the other. By
notebook 08, `block_counts()` will read like a table of contents for Shor's algorithm
and `gate_counts()` will be in the thousands.

## 2. `adjoint` — running a program backwards

Take any circuit you have written and ask: can it be undone? Not "can I write a circuit
that produces the same starting state" — can I take *this* circuit, whatever it does,
and run it in reverse to recover whatever state I fed it?

For a quantum circuit the answer is always yes, and the reason is a one-liner. Every
gate is a **unitary** matrix $U$, meaning $U^\dagger U = I$ where $U^\dagger$ is the
conjugate transpose. So every gate has an inverse, namely $U^\dagger$, which is itself a
legal gate. A circuit is a product of gates,
$$U = U_k \cdots U_2 U_1,$$
and the inverse of a product is the product of the inverses **in reverse order**:
$$U^{-1} = U_1^\dagger U_2^\dagger \cdots U_k^\dagger.$$
Undoing "put on socks, then shoes" is "take off shoes, then socks". Both halves matter —
reversing the order without inverting the gates, or inverting without reversing, gives
you something else entirely.

### Why classical computing does not get this for free

Look at a classical `AND` gate: two bits in, one bit out.

| $x$ | $y$ | $x \wedge y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

I tell you the output is $0$. Which input was it? You cannot say — three different
inputs produce it. Information that was there before is not there after; the gate is
*not injective*, so it has no inverse. Two bits went in and one came out, and the
missing bit did not go anywhere you can point to. (It went into the environment as
heat: this is **Landauer's principle**, and the bookkeeping is exact — erasing one bit
costs at least $k_B T \ln 2$ of dissipated energy. It is one of the places where
information theory and thermodynamics turn out to be the same subject.)

Quantum mechanics simply does not offer a gate like that. Unitary evolution is
invertible by construction, so **no information is ever lost** — it can be scrambled
beyond any hope of practical recovery, spread across a hundred qubits, but not deleted.
That constraint is the reason the `adjoint` scope below can exist at all, and in section
5 it will come back as a *cost*: to compute an `AND` on a quantum computer you must keep
enough around to run it backwards.

Here is what running backwards looks like. `scramble` is a deliberately arbitrary
seven-gate mess, including two rotations by angles, so that undoing it is not a
coincidence.

In [ ]:
@qsim.gate
def scramble(r: Register, *, theta: float) -> None:
    """A deliberately arbitrary seven-gate mess, so that undoing it means something."""
    H(r[0])
    Ry(r[1], theta=theta)
    CNOT(r[0], r[1])
    T(r[2])
    CNOT(r[1], r[2])
    Rz(r[0], theta=theta / 2)
    S(r[2])


qc = Circuit(name="undo", seed=1)
r = qc.register(3, name="q")
Ry(r[0], theta=1.1)  # an unremarkable starting state, so there is something to recover
Ry(r[2], theta=0.4)
before = qc.inspect.state_vector()

print("start:      ", qc.inspect.ket(max_terms=3))
scramble(r, theta=0.7)
print("scrambled:  ", qc.inspect.ket(max_terms=3))

with qc.adjoint():
    scramble(r, theta=0.7)

print("unscrambled:", qc.inspect.ket(max_terms=3))
print()
print("fidelity with the state we started from:", qc.inspect.fidelity(before))

Fidelity $1$ to fifteen digits. **Fidelity** was defined in notebook 03: it is
$|\langle\phi|\psi\rangle|^2$, the probability that a measurement asking "are you the
state $|\phi\rangle$?" answers yes. A value of $1$ means the state came back exactly,
not approximately.

Now look at what the library actually did. The block's ops carry the `scramble` tag, so
we can pull out the forward run and the reverse run and put them side by side:

In [ ]:
scramble_ops = [op.name for op in qc.history if op.block == "scramble"]
print("forward :", scramble_ops[:7])
print("backward:", scramble_ops[7:])

The second list is the first one reversed, with every gate replaced by its inverse.
`S` came back as `S†` and `T` as `T†` — those are real gates in the library, spelled
"S-dagger" and "T-dagger", and you will see them in histories and circuit diagrams.
`H` and `CNOT` came back unchanged, because each is its own inverse: doing either one
twice gets you back where you started. The rotations came back as `Ry` and `Rz` again,
with their angles negated — you cannot see that in the names, which is exactly why the
next cell prints the parameters.

### How the scope works, and why it has to

Notice that `with qc.adjoint():` had to see the *whole* block before it could run any of
it, since the last gate has to go first. So inside a scope, execution stops being eager:
the circuit switches to **record mode**, collecting operations into a buffer instead of
applying them. On exit the buffer is transformed — reversed and daggered — and only then
executed. This is the one exception to eager execution in qsim, and physics forces it:
you cannot reverse a list you have not finished reading.

The same two verbs exist one level down, on individual gates, which is what makes them
feel like one vocabulary rather than two features:

In [ ]:
print("H.adjoint() is H :", H.adjoint() is H)  # H is its own inverse
print("S.adjoint()      :", S.adjoint().label)
print("T.adjoint()      :", T.adjoint().label)
print()

qc = Circuit(seed=0)
q = qc.alloc("q")
Rz(q, theta=0.3)
Rz.adjoint()(q, theta=0.3)  # the *gate's* adjoint: applies -0.3
print("as recorded:", [(op.name, op.params) for op in qc.history])
print("final state:", qc.inspect.ket())

`Rz.adjoint()` did not need a new gate — rotating by $-\theta$ is the inverse of
rotating by $\theta$, so the adjoint of a parametrized gate negates its angle. The
history records `-0.3`, and the qubit is back at $|0\rangle$.

Blocks get the same two methods, so you can write `scramble.adjoint()(r, theta=0.7)`
instead of opening a scope. Use whichever reads better: the scope form is for "undo
these few lines", the method form for "here is a named subroutine, and here is its
inverse", which is how the QFT will be used in notebook 07.

In [ ]:
qc = Circuit(seed=2)
r = qc.register(3, name="q")
Ry(r[0], theta=1.1)
before = qc.inspect.state_vector()

scramble(r, theta=0.7)
scramble.adjoint()(r, theta=0.7)

print("fidelity:", qc.inspect.fidelity(before))
print("block_counts():", qc.block_counts())

### What cannot be undone

There is exactly one operation in this library that is not a unitary, and you have known
about it since notebook 01: **measurement**. It does not scramble information, it
destroys it — the superposition it reports on is gone afterwards, and the outcome it
reports was not determined beforehand. There is nothing to replay backwards.

So what happens if you put one inside a scope?

In [ ]:
qc = Circuit(seed=0)
q = qc.alloc("q")
try:
    with qc.adjoint():
        H(q)
        qc.measure(q)
except QsimError as err:
    print(err)

> cannot measure q inside a combinator scope. Measurement is the one irreversible
> operation in the library: it destroys the superposition it reports on, so there is
> nothing to replay backwards and no way to condition it on a control that is itself in
> superposition. Every other operation here is a unitary, and unitaries can always be
> inverted and controlled — that is exactly why the scopes work. Measure after the scope
> has closed.

The refusal covers the control scope too, and for a second reason worth spelling out.
Conditioning a measurement on a control that is in superposition would mean the
measurement both happened and did not happen — and a measurement is precisely the thing
that *cannot* be in superposition with its own absence, because it produces a definite
classical outcome. Notebook 06 will complicate this claim considerably, by showing that
"measurement" is itself just entanglement with something big. For now: measure after the
scope has closed.

## 3. `control` — doing something "only if", where the "if" is in superposition

`with qc.control(c):` runs the body only in the branches where `c` is $|1\rangle$. If
`c` is $|0\rangle$ the body does nothing; if `c` is $|1\rangle$ it happens in full. That
much is an ordinary `if` statement.

The interesting case is the one an `if` statement cannot express. Put `c` in
$|+\rangle = (|0\rangle + |1\rangle)/\sqrt2$ first, so that it is *both* — and then the
result is not "the block ran" or "the block did not run", and it is not a coin flip
between them either. It is a superposition of the two.

In [ ]:
qc = Circuit(name="maybe-bell", seed=0)
c = qc.alloc("c")
a = qc.alloc("a")
b = qc.alloc("b")

H(c)  # c is now |+⟩ — both answers at once
with qc.control(c):
    bell(a, b)

print(qc.inspect.ket())

Read that out loud, remembering the qubit order is `c a b`:

$$|\psi\rangle
 = \tfrac{1}{\sqrt2}\,|0\rangle_c \otimes |00\rangle_{ab}
 \;+\; \tfrac{1}{\sqrt2}\,|1\rangle_c \otimes
   \tfrac{1}{\sqrt2}\big(|00\rangle + |11\rangle\big)_{ab}$$

which is what the printed amplitudes $0.707, 0.5, 0.5$ say. Half the amplitude sits in a
branch where `a` and `b` are untouched in $|00\rangle$; the other half sits in a branch
where they are a Bell pair. **The circuit itself is in superposition of having run and
not having run**, and `c` is entangled with the answer to "did it run?".

This is not a curiosity. It is the engine of essentially every quantum algorithm in the
second half of this project: phase estimation controls a unitary on a register of
superposed control qubits, Shor's algorithm controls modular multiplication the same way.
"Do a different amount of work in every branch, then interfere the branches" is the whole
game.

In [ ]:
fig = viz.amplitudes(qc, phase_as_hue=False)

In [ ]:
print("entropy of c with the rest:", round(qc.inspect.entanglement_entropy([c]), 4), "bits")
print("entropy of a,b as a pair  :", round(qc.inspect.entanglement_entropy([a, b]), 4), "bits")
print()
for op in qc.history:
    print(f"  {op.name:<5} targets={op.qubit_ids}  controls={op.controls}  block={op.block!r}")

Two things to notice.

The entropy is $0.60$ bits, not the full $1$ bit a Bell pair gives. Entanglement entropy
measures how *distinguishable* the branches are, and here they are not perfectly
distinguishable: the "did not run" branch is $|00\rangle$ and the "did run" branch is
$(|00\rangle + |11\rangle)/\sqrt2$, which still contains $|00\rangle$. The two overlap,
so `c` is only partly correlated with what happened. Entropy is a continuous quantity,
and this is what a partial record looks like — a fact section 4 leans on hard.

And look at the history: `bell`'s two gates came out with `c`'s id prepended to their
controls. `H` became a controlled-$H$; `CNOT`, which already had one control, now has
two. **Nothing was decomposed.** At the level of the state tensor, "conditioned on all
controls being $|1\rangle$" means *slice the axes where those controls read 1 and apply
the gate only there* — one more slice per control, at no extra conceptual cost.

Real hardware is not so lucky. A physical machine has one- and two-qubit gates and
nothing else, so a three-control gate must be *compiled* into a sequence of them,
typically dozens, sometimes with extra scratch qubits. That compilation is a large part
of what makes quantum programs expensive. The slice qsim performs is the mathematical
meaning those decompositions work to reproduce — which is why this notebook can show you
the meaning first and leave the cost to a compiler.

Because controls are just accumulated, nesting scopes and listing several controls are
the same thing:

In [ ]:
qc = Circuit(seed=0)
c1, c2, t = qc.alloc_many(3)
X(c1)
X(c2)  # both controls are |1⟩, so the body should fire

with qc.control(c1):
    with qc.control(c2):
        X(t)

print(qc.inspect.ket())
print([(op.name, op.controls, op.qubit_ids) for op in qc.history])

`with qc.control(c1): with qc.control(c2): X(t)` recorded a single `X` with controls
`(0, 1)` — which is exactly `qc.control(c1, c2)`, and exactly the `Toffoli` gate.

One thing you may not do: control a block on a qubit the block itself acts on.

In [ ]:
qc = Circuit(seed=0)
c = qc.alloc("c")
t = qc.alloc("t")
try:
    with qc.control(c):
        X(t)
        H(c)  # c is the control — it cannot also be a target
except NoCloningError as err:
    print(err)

> c controls this block and is also acted on inside it. A qubit cannot control an
> operation on itself: deciding what to do to it would mean reading its value, and the
> result would be a copy of that value. Use a separate qubit as the control.

This is the no-cloning theorem from notebook 02, showing up as a scoping rule. Note also
what the failure did *not* do: the `X(t)` recorded before the bad line never ran, because
recorded ops are only executed on a clean exit. A scope that raises leaves the state
untouched rather than half-transformed.

## 4. Ancillas — scratch space, and the trap

An **ancilla** is a scratch qubit: one you borrow in $|0\rangle$, use as working memory,
and give back. Classical code does this constantly and thinks nothing of it — allocate a
temporary, use it, let it fall out of scope, and the garbage collector takes care of the
rest.

Quantum mechanics charges for it, and the bill is surprising.

```python
with qc.ancilla(2) as scratch:    # two fresh qubits in |00⟩
    ...                           # use them
    ...                           # then put them back to |00⟩ yourself
```

On exit, the scope **checks numerically** that the scratch qubits are back in
$|0\dots0\rangle$ *and* unentangled with everything else, and raises `DirtyAncillaError`
if they are not. There is no way to switch the check off, and there is no way to release
a qubit by dropping its handle instead. To see why that is a physical requirement and
not a style preference, we need to build the failure.

### A two-slit experiment with one qubit

Take a single qubit in $|0\rangle$ and apply $H$ twice in a row. $H$ is its own inverse,
so the qubit ends in $|0\rangle$ with certainty. But the interesting way to say the same
thing is:

- the first $H$ opens **two paths**, $|0\rangle$ and $|1\rangle$, each with amplitude
  $1/\sqrt2$;
- the second $H$ recombines them, and the two ways of arriving at $|1\rangle$ have
  amplitudes $+\tfrac12$ and $-\tfrac12$, which **cancel**.

That cancellation is interference, and it is the entire reason the answer is certain
rather than a coin flip. This is a two-slit experiment with the slits made of amplitudes.
Watch the coherence — the off-diagonal entry $\rho_{01}$ of the qubit's reduced density
matrix from notebook 02, which is what "still in superposition" looks like numerically:

In [ ]:
def report(qc: Circuit, q: Qubit) -> None:
    """Print everything worth knowing about q on its own."""
    rho = qc.inspect.reduced_density_matrix([q])
    print(f"    coherence |ρ01|           = {abs(rho[0, 1]):.3f}")
    print(f"    entanglement entropy of q = {qc.inspect.entanglement_entropy([q]):.3f} bits")
    print(f"    P(q reads 0)              = {rho[0, 0].real:.3f}")


qc = Circuit(name="two-slit", seed=0)
q = qc.alloc("q")

H(q)
print("halfway — two paths open:")
report(qc, q)

H(q)
print("after the second H — the paths recombine:")
report(qc, q)

Halfway through, the qubit is a fifty-fifty superposition with coherence $0.5$ — the
largest it can be. At the end, $P(0) = 1$: the outcome is certain. Nothing is entangled
with anything, and the entropy is zero throughout.

(The coherence reads $0$ at the end too, but for the opposite reason: $|0\rangle$ is not
a superposition of anything, so it has no off-diagonal term. The coherence is a snapshot
of the *middle* of the experiment. Where interference shows up at the end is in
$P(0) = 1$ rather than $P(0) = 0.5$.)

### Now let a scratch qubit watch

Borrow one ancilla and copy `q`'s value into it with a `CNOT`, in between the two $H$s.
This is the most innocuous thing imaginable — a scratch variable, holding a value we
might want later. We will not look at it. We will not measure it. We will not condition
anything on it. It simply sits there.

In [ ]:
qc = Circuit(name="recorded", seed=0)
q = qc.alloc("q")
try:
    with qc.ancilla(1) as scratch:
        H(q)
        CNOT(q, scratch[0])  # the scratch qubit now records which path q took
        print("halfway, with a which-path record on the scratch qubit:")
        report(qc, q)

        H(q)
        print("after the second H:")
        report(qc, q)
except DirtyAncillaError as err:
    print()
    print(err)

The coherence is **gone** — $|\rho_{01}| = 0$ the instant the `CNOT` lands, before the
second $H$ is even applied. And so the second $H$ has nothing to interfere: the paths
arrive and add up in probability rather than in amplitude, and the answer that used to be
certain is now a fair coin, $P(0) = 0.5$.

Then the scope exits and refuses:

> anc0 are not in |0>: probability 0.5 of finding a 1. Scratch qubits must be uncomputed
> back to |0> before release, not merely ignored. Leftover entanglement is a *record* of
> which branch of the computation happened, and a branch that has been recorded can no
> longer interfere with the others — which is where a quantum algorithm's advantage comes
> from. Undo the operations that dirtied these qubits, in reverse order.

Sit with what just happened, because it is the single most important idea in this
notebook and it is not a software issue at all.

**Nobody looked.** No measurement was performed, no `if` branched on the scratch qubit,
no human read a dial. The mere *existence* of a record — one extra qubit, somewhere, that
is correlated with which path `q` took — was enough to destroy the interference. The
entanglement entropy of `q` went from $0$ to $1$ bit, meaning `q` no longer has a state
of its own; it is one half of an entangled pair, and half of an entangled pair is a
statistical mixture, and mixtures do not interfere.

Why? Because interference is amplitudes adding, and amplitudes only add when the paths
being combined are *indistinguishable*. Once the scratch qubit holds a different value on
each path, the two paths end in different total states — $|0\rangle_q|0\rangle_{anc}$
versus $|1\rangle_q|1\rangle_{anc}$ — and different states cannot cancel. It does not
matter whether anyone ever reads the ancilla. What matters is that the information is
*there*, in principle available, distinguishing the paths.

This is the whole of notebook 06 in miniature. Swap "one scratch qubit" for "$10^{23}$
air molecules and photons" and you have **decoherence**: the reason the world you live in
looks classical.

### Uncomputation puts it back

Since the damage was done by a `CNOT`, it can be undone by a `CNOT` — the gate is its own
inverse. Run it again *before* the paths recombine, and the ancilla is returned to
$|0\rangle$ regardless of which path `q` is on, so the record is erased and the two paths
become indistinguishable again.

In [ ]:
qc = Circuit(name="uncomputed", seed=0)
q = qc.alloc("q")
with qc.ancilla(1) as scratch:
    H(q)
    CNOT(q, scratch[0])  # compute: write the record
    CNOT(q, scratch[0])  # uncompute: erase it again, before recombining
    H(q)
    print("with the record erased before the paths recombine:")
    report(qc, q)

print()
print("after the scope closes:", qc.inspect.ket(), " — n_qubits =", qc.n_qubits)

Interference restored. The three numbers are identical to the no-scratch run — entropy
$0$, and the certain outcome $P(0) = 1$ — and the scope exits without complaint. The
circuit is down to one qubit again — the ancilla's axis was
removed from the state tensor, which is only safe *because* the check confirmed all the
amplitude was in the $|0\rangle$ slice. Reaching for `scratch[0]` after this point raises
`DeadQubitError`; the handle no longer names anything physical.

Notice that the check is a single number. If the probability of finding every ancilla at
$0$ is exactly $1$, then every amplitude with any ancilla bit set is zero, so the state
factorizes as $|0\dots0\rangle_{anc} \otimes |\text{rest}\rangle$ — the ancillas are
simultaneously *in* $|0\rangle$ **and** *unentangled*. One quantity checks both, and a
real quantum computer could not compute it. Verifying uncomputation is a simulator
superpower, and it is why this library can teach the discipline instead of just
recommending it.

### But then what was the ancilla *for*?

Fair objection: a `CNOT` followed by its own inverse is a very expensive way to do
nothing. So let the scratch qubit do some real work in between — here, a $R_z(\theta)$
rotation, which puts a phase on the ancilla's $|1\rangle$ component.

That phase belongs to the ancilla. But the ancilla's value is correlated with `q`'s, so
the phase lands on `q`'s $|1\rangle$ branch too — the effect known as **phase kickback**,
which notebook 07 will build phase estimation out of. Uncomputing the `CNOT` then cleans
up the record while leaving the phase behind, and the final $H$ turns that phase into a
shifted probability.

In [ ]:
print(f"{'theta':>7}   {'P(q reads 0)':>12}   {'cos²(θ/2)':>10}   {'entropy':>7}")
for theta in [0.0, np.pi / 4, np.pi / 2, 2 * np.pi / 3, np.pi]:
    qc = Circuit(seed=0)
    q = qc.alloc("q")
    with qc.ancilla(1) as scratch:
        H(q)
        CNOT(q, scratch[0])            # compute: correlate the scratch with q
        Rz(scratch[0], theta=theta)    # work: a phase, applied to the scratch qubit
        CNOT(q, scratch[0])            # uncompute: erase the correlation
        H(q)
        rho = qc.inspect.reduced_density_matrix([q])
        entropy = qc.inspect.entanglement_entropy([q])
    print(f"{theta:>7.4f}   {rho[0, 0].real:>12.6f}   {np.cos(theta / 2) ** 2:>10.6f}"
          f"   {entropy:>7.4f}")

$P(0) = \cos^2(\theta/2)$ exactly, and the entropy is $0$ in every row — the ancilla is
clean each time, so every scope exits happily. The scratch qubit did real work: it
carried a phase that `q` could not have acquired on its own, and then it was tidied away
without leaving a trace. That is the pattern every algorithm in this project uses.

Compare the two experiments and the moral is sharp. **The scratch qubit is allowed to
affect the answer. It is not allowed to remember.**

### A record can be partial

Nothing here is all-or-nothing. Replace the `CNOT` — which copies `q`'s value perfectly —
with a *partial* copy: rotate the scratch qubit by an angle $\varphi$, but only in the
branch where `q` is $|1\rangle$. At $\varphi = \pi$ that is exactly a `CNOT`; at
$\varphi = 0$ it does nothing; in between, the scratch qubit ends up in two states that
are neither identical nor orthogonal, so it holds a fuzzy, partial record of the path.

Note how it is written: `with qc.control(q):` around an ordinary `Ry`. Section 3's tool,
used to build a section 4 experiment.

In [ ]:
def partial_record(phi: float) -> tuple[float, float]:
    """The two-slit sandwich, with a scratch qubit that learns only *part* of the path."""
    qc = Circuit(seed=0)
    q = qc.alloc("q")
    s = qc.alloc("s")

    H(q)
    with qc.control(q):
        Ry(s, theta=phi)  # rotate the scratch qubit, but only in q's |1⟩ branch
    entropy = qc.inspect.entanglement_entropy([q])

    H(q)  # try to recombine the paths
    p0 = float(qc.inspect.reduced_density_matrix([q])[0, 0].real)
    return p0, entropy


phis = np.linspace(0.0, np.pi, 121)
p0s, entropies = zip(*[partial_record(float(p)) for p in phis])

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(phis, p0s, linewidth=2, color="tab:blue")
ax.set_xlabel("φ — how far the scratch qubit turns in q's |1⟩ branch")
ax.set_ylabel("P(q reads 0) after recombining", color="tab:blue")
ax.set_ylim(0.45, 1.03)
ax.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi])
ax.set_xticklabels(["0", "π/4", "π/2", "3π/4", "π  (a full CNOT)"])
ax.set_title("the more the scratch qubit knows, the less the paths interfere")

ax2 = ax.twinx()
ax2.plot(phis, entropies, linewidth=2, color="crimson", linestyle="--")
ax2.set_ylabel("entanglement entropy of q (bits)", color="crimson")
ax2.set_ylim(-0.03, 1.03)

print("φ = 0    :  P(0) =", round(p0s[0], 4), "  entropy =", round(entropies[0], 4))
print("φ = π    :  P(0) =", round(p0s[-1], 4), "  entropy =", round(entropies[-1], 4))

The two curves are two views of one quantity. As $\varphi$ grows the scratch qubit
becomes a better and better witness — the entropy of `q` climbs from $0$ to a full bit —
and in exact step with that, the certainty of the outcome decays from $P(0) = 1$ to
$P(0) = 0.5$, a fair coin. There is no threshold at which interference "switches off".
Information leaking away and interference fading are the same process, measured on two
dials.

This trade has a name in the literature — *complementarity*, or the
"which-path / interference" duality — and it is usually stated as a sharp inequality
between distinguishability and fringe visibility. Here it is just what the numbers do.

## 5. Bennett's trick: compute, copy out, uncompute

Now put section 2 and section 4 together, because the constraint from one is solved by
the technique from the other.

Section 2: quantum gates are reversible, so you cannot write an `AND` that throws a bit
away. The reversible stand-in is the **Toffoli** gate, which takes three qubits and flips
the third one exactly when the first two are both $|1\rangle$ — so with the third
starting at $|0\rangle$, it leaves $x$, $y$, and $x \wedge y$. Nothing is destroyed,
because the inputs are still there. That is the price: reversible computing does not
delete, it accumulates.

Section 4: accumulated intermediate results are not free, and they are not merely
untidy. Every leftover scratch qubit is a *record*, and records kill interference. A
long computation done naively leaves a trail of them, and by the end there is nothing
left to interfere.

**Charles Bennett's insight (1973)** resolves the tension in three steps:

1. **Compute.** Run the whole calculation, letting the scratch qubits fill with whatever
   intermediate junk they fill with.
2. **Copy out.** Write the answer — and only the answer — into an output register, with
   `CNOT`s. This is not cloning: it copies one *bit* out of a basis state, not an unknown
   superposition, which is a permutation and perfectly legal.
3. **Uncompute.** Run step 1 backwards. Every intermediate result unwinds, and the
   scratch qubits return to $|0\rangle$ — clean, unentangled, and correlated with
   nothing.

You pay roughly double the gates and you keep the interference. Here is the smallest
honest example: a reversible `AND` on inputs that are in superposition, so that there is
interference to lose.

In [ ]:
qc = Circuit(name="reversible-and", seed=0)
x, y, out = qc.alloc_many(3)
H(x)
H(y)  # all four inputs (x, y) at once
print("before:", qc.inspect.ket())

with qc.ancilla(1) as scratch:
    Toffoli(x, y, scratch[0])  # 1. compute: scratch = x AND y
    CNOT(scratch[0], out)      # 2. copy the answer out
    Toffoli(x, y, scratch[0])  # 3. uncompute: scratch back to |0⟩

print("after :", qc.inspect.ket())
print()
print("n_qubits =", qc.n_qubits, " gate_counts:", qc.gate_counts())

Read the output as `x y out`. Four equally weighted branches, and in each one `out` holds
$x \wedge y$: $|000\rangle, |010\rangle, |100\rangle$ and — the only branch where both
inputs are $1$ — $|111\rangle$. The whole truth table, computed in superposition, in one
pass. And the scope exited cleanly: the scratch qubit is gone, back to three qubits.

Two Toffolis where a classical machine needs one `AND`. That factor of two is the
standing tax of reversible computing, and it buys something specific: **nothing is left
recording the branches except the answer itself.**

That qualification is the whole subtlety. `out` certainly records something — it holds
$x \wedge y$, which is the entire point of running the circuit, and it does distinguish
the $|11\rangle$ branch from the other three. Uncomputation is not about erasing that.
What step 3 removes is the **scratch**: a second, redundant copy of the same information,
sitting in a qubit nobody will ever look at again.

Leave it there and the circuit still computes the right answer in every branch. What it
loses is the ability of those branches to *recombine* later. If you measure straight away,
leftover scratch costs you nothing. But an algorithm that works by bringing branches back
together so their amplitudes can cancel — which is what phase estimation, Shor's and
Grover's all do — finds the branches it needs to interfere have been tagged as different,
and things that are distinguishable do not interfere.

If you want to see that failure with your own eyes rather than take it on faith, that is
test T18 in Phase 5: Shor's algorithm run with the uncomputation deliberately omitted.
The period-finding peaks smear into a flat distribution. Nothing errors; the answer is
simply not there any more.

This is why Phase 4 — the reversible adders, modular multiplication, and modular
exponentiation that Shor's algorithm runs on — is written as compute / copy out /
uncompute at every level of its recursion, and why every one of those levels uses
`with qc.ancilla(...)`. The check at the end of that block is what will tell you the
arithmetic is honest.

## 6. Conjugation: do, act, undo

Here is a shape you have already written several times without naming it. Section 4's
two-slit experiment was $H$, something, $H$. Section 4's uncomputation was `CNOT`,
something, `CNOT`. The eraser in notebook 06 will be a coupling, something, the
coupling backwards.

The general form is
$$V\,U\,V^\dagger$$
— do $V$, do $U$, undo $V$ — and it is called **conjugation**. It is the single most
common composite in quantum programming, for one reason: $U$ is usually easy to write in
*some* basis, and $V$ is the change into that basis.

- Dephasing a qubit along the $x$ axis is ordinary dephasing wrapped in $H$.
- Grover's oracle (notebook 09) is a controlled-$Z$ wrapped in $X$ gates.
- The Fourier-space adder (Phase 4) is a handful of phase rotations wrapped in a QFT.

You can write the wrapper twice by hand, and everything up to now has. The risk is that
the two halves drift apart when the code is edited — and the bug that produces is not an
error message, it is a subtly wrong answer. So conjugation gets a combinator:

```python
with qsim.within(H, q):        # V = H(q), applied right now
    dephasing_coupling(...)    # the body, run eagerly
                               # H(q) applied again on the way out
```

Two details distinguish it from the scopes in sections 2 and 3.

**Only $V$ is recorded.** The body is *not*. `within` runs $V$ under a private buffer
just long enough to learn which gates it consists of, emits them, and then gets out of
the way: the body runs eagerly, and you can inspect the state between any two of its
lines, exactly as outside a scope. Compare `control`, which has no choice but to record
its body — "run this only where `c` is $|1\rangle$" is a counterfactual, so the ops must
be *rewritten* before they run, and you cannot rewrite an op you have not seen yet.
"Undo $V$ afterwards" needs $V$ remembered and nothing else.

**$V$ may not measure**, for the obvious reason: the whole construct rests on being able
to undo $V$ on the way out.

Start with what conjugation is *for*. Take a qubit in $|+\rangle$ and let a single
environment qubit record it — once in the ordinary computational basis, once wrapped
in $H$.

In [ ]:
from qsim.decoherence import dephasing_coupling


def dephase_a_plus_state(in_the_x_basis: bool) -> tuple[float, float]:
    """Prepare |+⟩, let one environment qubit record it, and report the damage."""
    qc = Circuit(seed=0)
    q = qc.alloc("q")
    env = qc.environment(1)
    H(q)  # |+⟩ = (|0⟩ + |1⟩)/√2

    if in_the_x_basis:
        with qsim.within(H, q):  # V: rotate |+⟩,|−⟩ onto |0⟩,|1⟩ — and back on exit
            dephasing_coupling(q, env[0], theta=np.pi)  # U: the environment records
    else:
        dephasing_coupling(q, env[0], theta=np.pi)

    return qc.inspect.coherence(q), qc.inspect.system_entropy()


for label, in_x in [("z basis (plain dephasing)", False),
                    ("x basis (conjugated by H)", True)]:
    coherence, entropy = dephase_a_plus_state(in_x)
    print(f"{label:>26} :  coherence = {coherence:.4f}   entropy = {entropy:.4f}")

Same coupling, same strength, same qubit — and in one case the superposition is destroyed
and in the other it is not touched at all.

`dephasing_coupling` writes into the environment the answer to "is `q` in $|0\rangle$ or
$|1\rangle$?". Our qubit is $|+\rangle$, which is an even mixture of both, so that
question has no definite answer: asking it forces the environment to become entangled
with `q`, and the superposition dies. Entropy $1$ bit, coherence gone.

Wrap the same coupling in $H$ and the question changes. $H$ maps $|+\rangle \to
|0\rangle$ and $|-\rangle \to |1\rangle$, so in the rotated frame the environment is
asking "is `q` in $|+\rangle$ or $|-\rangle$?" — and *that* question already has a
definite answer. Nothing is learned that was not already true, no entanglement forms,
and the second $H$ puts everything back. Entropy $0$, coherence untouched.

**A basis change did not change what the environment can do; it changed what it asks
about.** That is the whole content of the sandwich, and it is why the pattern is
everywhere.

### The algebra you get for free

Because `within` records $V$ rather than trusting you to retype it, a block built with it
behaves properly under the other combinators. Watch what `.adjoint()` does to a sandwich.

In [ ]:
@qsim.gate
def x_dephasing(q: Qubit, env: Qubit, *, theta: float) -> None:
    """Dephasing in the |+⟩/|−⟩ basis — the same three lines, given a name."""
    with qsim.within(H, q):
        dephasing_coupling(q, env, theta=theta)


def bloch_report(qc: Circuit, q: Qubit, label: str) -> None:
    x, y, z = qc.inspect.bloch_vector(q)
    print(f"{label:>22} :  bloch = ({x:+.3f}, {y:+.3f}, {z:+.3f})"
          f"   entropy = {qc.inspect.system_entropy():.4f}")


qc = Circuit(name="sandwich", seed=0)
q = qc.alloc("q")
env = qc.environment(1)
Ry(q, theta=0.9)  # a lopsided state, tilted away from every axis
bloch_report(qc, q, "start")

x_dephasing(q, env[0], theta=np.pi)
bloch_report(qc, q, "after the sandwich")
print("   ops:", [(op.name, op.params) for op in qc.history[1:4]])

x_dephasing.adjoint()(q, env[0], theta=np.pi)
bloch_report(qc, q, "after its adjoint")
print("   ops:", [(op.name, op.params) for op in qc.history[4:]])
print()
print("blocks:", qc.block_counts())

Three things happened there, and each is worth a sentence.

**The Bloch vector lost its $z$ but kept its $x$.** The Bloch vector, from notebook 01,
is the three numbers $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ that
describe a single qubit's state as a point in a ball. Dephasing conjugated by $H$ leaves
the $x$ component untouched and flattens everything perpendicular to it — because in the
rotated basis, "$x$" *is* the computational axis, and plain dephasing never moves that
one. Which states survive a coupling is decided by the coupling, not by the states.
That is **einselection**, and notebook 06 §8 is about nothing else.

**The adjoint inverted only the filling.** Compare the two op lists: forward is
`H, Ry(π), H`, backward is `H, Ry(−π), H`. The wrapper is in the same place both times.
That is the identity $(VUV^\dagger)^\dagger = V U^\dagger V^\dagger$, and you get it for
free: reversing the whole three-op sequence puts $V$ back at the front and $V^\dagger$
back at the end, because $V$ and $V^\dagger$ swap places *and* swap roles. You never have
to think about it, which is the point of writing the sandwich with a combinator.

**The coherence came back exactly.** The environment had a perfect record of `q`, and
running the coupling backwards unwrote it. That is a **quantum eraser** in five lines,
and it works because nothing was measured — the "record" was an entangled qubit, not a
classical outcome. Section 7 makes that distinction the whole subject.

One more thing this buys, in the library rather than in your code: `pointer_coupling` in
`qsim.decoherence` — the coupling notebook 06 uses to choose which basis the environment
listens in — *is* `dephasing_coupling` wrapped in a `within`. Its source says
"this is dephasing, conjugated into another basis" in its structure, not only in its
comment. The identity above is why that is safe to write once and control, invert and
nest freely.

## 7. The tape

`qc.history` has been in the background of this whole notebook: a list of every operation
that ran, which we have printed to check what `adjoint` and `control` did. It is time to
say what it actually is.

**The history is a tape, and it is the same idea deep-learning libraries are built on.**
PyTorch, JAX and the rest run each operation immediately — exactly as qsim does — and
*also* record it, because a recorded sequence can be walked backwards later. That is what
"backpropagation" walks: a tape of the operations that produced a number, replayed in
reverse, each step replaced by its derivative. (If you have never used one of those
libraries, nothing is lost; the point is only that "run eagerly, keep a record, transform
the record later" is a known and useful shape, not something invented here.)

Our tape is the same shape with one part *simpler*. Autograd has to save the intermediate
values as it goes, because a derivative depends on where you evaluate it. We save
nothing. Every gate is invertible, so the tape alone is enough to get back — "no saved
values" is the software shadow of "unitary evolution destroys no information".

Two methods make the tape usable rather than merely readable:

```python
mark = qc.checkpoint()    # remember this position on the tape
...                       # do some work
qc.rewind(mark)           # execute the inverse of everything since the mark
```

`checkpoint()` copies nothing — it is a position, plus a note of which qubits existed
there. `rewind(mark)` walks the ops recorded since the mark, newest first, and runs each
one's inverse.

Watch it undo decoherence. `dephasing_coupling` is the coupling from notebook 06, and at
$\theta = \pi$ it writes a perfect record of `q` into an environment qubit — which, as
section 4 taught us, destroys `q`'s superposition without anybody looking at anything.

In [ ]:
qc = Circuit(name="tape", seed=0)
q = qc.alloc("q")
env = qc.environment(1)
H(q)  # |+⟩ — a superposition, with coherence 0.5

mark = qc.checkpoint()
print("at the mark    :  coherence =", round(qc.inspect.coherence(q), 4),
      "  entropy =", round(qc.inspect.system_entropy(), 4))

dephasing_coupling(q, env[0], theta=np.pi)  # let the environment record q, perfectly
print("after coupling :  coherence =", round(qc.inspect.coherence(q), 4),
      "  entropy =", round(qc.inspect.system_entropy(), 4))

qc.rewind(mark)
print("after rewind   :  coherence =", round(qc.inspect.coherence(q), 4),
      "  entropy =", round(qc.inspect.system_entropy(), 4))
print()
print("history :", [op.name for op in qc.history])
print("counts  :", qc.gate_counts())

Coherence $0.5 \to 0 \to 0.5$, entropy $0 \to 1 \to 0$. The environment learned which
branch `q` was in, the superposition died of the learning, and then the learning was
undone and the superposition came back — exactly.

Now look at the history, because it is the part that surprises people:

```
history : ['H', 'Ry', 'Ry']
counts  : {'H': 1, 'Ry': 2}
```

**The rewind is on the tape.** The undo did not delete the coupling from the record; it
appended the gate that reverses it. The *state* went back, and the *record* says how it
got back — the way an editor's undo appears in the edit log rather than erasing your
keystrokes from it.

That is a deliberate choice, and the alternative was rejected on purpose. A tape that
quietly deleted the last few entries would make `gate_counts()` lie, and gate counts are
what a circuit costs to run: those inverse gates take time on real hardware and give the
qubits more time to decohere. A diagram drawn from a truncated tape would be a diagram of
a program nobody ran. The tape is a record of what happened, and it is never rewritten.

One consequence to know before you put `rewind` in a loop: rewinding twice to the *same*
mark also undoes the first rewind, since the first rewind is itself on the tape. The
state still comes out right — an undo of an undo is a redo, which is then undone in turn
— but the op count doubles each pass. When sweeping a parameter, take a fresh
`checkpoint()` after each `rewind`, and the tape grows by one entry per pass.

### The operation that is not on the tape

Everything above works because every gate carries a rule for its own inverse. There is
one operation that does not.

In [ ]:
qc = Circuit(name="severed", seed=0)
q = qc.alloc("q")
env = qc.environment(1)
mark = qc.checkpoint()

H(q)
CNOT(q, env[0])  # a coherent record — a gate, and so undoable
print("q read:", qc.measure(q))  # a classical outcome — and not undoable

try:
    qc.rewind(mark)
except QsimError as err:
    print()
    print(err)

Read that message slowly, because it is the sharpest statement of the difference between
the two ways a qubit can lose its superposition.

> The measurement severed the tape, exactly the way a non-differentiable operation severs
> an autograd graph — everything before the cut is intact, but you cannot get back
> through it.

A gate can always be undone: the tape says what it was, and every gate carries a rule for
its own inverse. A measurement cannot, and not because the library declines to try.
Measurement *discarded* the branches it did not report, and there is no operation that
brings back a discarded branch. The tape ends there.

But notice what the first two lines of that cell did. `CNOT(q, env[0])` also produced a
record of `q` — a which-path record, exactly like section 4's scratch qubit — and *that*
record was written by a gate. Left alone, it destroys interference just as thoroughly as
a measurement does. Undo it, and the interference comes back exactly, as section 6's
`x_dephasing.adjoint()` already showed.

So the distinction is not "measured" versus "not measured". It is:

- **a coherent record** — written by a unitary into another qubit, still on the tape,
  still invertible. Decoherence, and undoable.
- **a classical outcome** — a branch chosen, the rest thrown away, nothing on the tape.
  Measurement, and final.

Keep the record coherent instead of measuring, and you can rewind through it. That is
the **quantum eraser**, and it is section 9 of `06-decoherence.ipynb`. Notebook 06 also
takes the next step and asks why, in a laboratory rather than a simulator, the second
case is overwhelmingly the normal one: not because measurement is a different kind of
physics, but because a record spread over $10^{23}$ air molecules is one that nobody is
ever going to run backwards.

## 8. Watching a program run

Every plot in this notebook so far was made by stopping the circuit at a chosen moment
and asking the Inspector a question. That works, but it means editing the program you
wanted to observe — and a program you have edited to watch it is not quite the program
you were watching.

`qc.on_op(fn)` attaches an **observer** instead. Your function is called after every
operation the circuit runs, with two arguments: the `Op` that just ran, and the circuit
itself, whose state is already updated. Nothing about the program under observation
changes.

```python
handle = qc.on_op(fn)     # fn(op, circuit), called after every op
handle.remove()           # stop listening
```

Here is an entropy trace: one entanglement-entropy reading per gate, collected while a
circuit entangles three qubits and then unentangles them again.

In [ ]:
qc = Circuit(name="traced", seed=0)
a, b, c = qc.alloc_many(3)

entropies: list[float] = []
labels: list[str] = []


def watch(op: qsim.Op, circuit: Circuit) -> None:
    """Called after every op, with the state already updated."""
    entropies.append(circuit.inspect.entanglement_entropy([a]))
    labels.append(op.name)


handle = qc.on_op(watch)

H(a)  # spread a's superposition across all three qubits...
CNOT(a, b)
CNOT(b, c)
T(c)  # ... do a little work at the far end ...
CNOT(b, c)  # ... and put it all back
CNOT(a, b)
H(a)

handle.remove()
X(a)  # not traced: the hook is gone

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(range(1, len(entropies) + 1), entropies, marker="o", linewidth=2, color="crimson")
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.set_xlabel("op, in the order the circuit ran them")
ax.set_ylabel("entanglement entropy\nof a (bits)")
ax.set_ylim(-0.05, 1.1)
ax.set_title("one number per gate, collected by a hook")

print("ops the hook heard about:", labels)
print("ops the circuit ran     :", [op.name for op in qc.history])

The trace is the story of the circuit in one line. Entropy climbs to a full bit as the
`CNOT`s spread `a`'s superposition across all three qubits, sits there while the `T`
does its work, and comes back to zero as the `CNOT`s are undone — the uncomputation of
section 4, watched live instead of checked at the end. The `X` after `handle.remove()`
is missing from the plot, because by then nobody was listening.

Three things about hooks are worth stating plainly:

- **A hook sees measurements too.** `op.result` is `None` for a gate and $0$ or $1$ for
  a measurement, so a hook that only cares about measurements filters on it itself.
  There is deliberately no filtering machinery in the library.
- **A hook may not apply gates.** Emitting an operation from inside a hook raises: the
  op would land on the tape with no line of your program accounting for it, and it
  would immediately fire every hook again, including the one that emitted it. Hooks
  watch; combinators transform.
- **This is not a special feature.** `viz.entropy_trace` in Phase 6 will be exactly the
  cell above with nicer axes — attach a hook, run the circuit, plot the list. Anything
  the Inspector can compute can be traced this way, one number per gate, with no
  changes to the program being traced.

And that is the last of the four ideas. Blocks name a piece of a program; `adjoint` and
`control` transform one; `within` wraps one; and `checkpoint`, `rewind` and `on_op` treat
the record of one as an object in its own right.

## What you now know

- **`@qsim.gate`** turns a function of qubit handles into a reusable **block**. Calling
  it runs eagerly; the history stores elementary gates tagged with the block they came
  from. `block_counts()` says what the circuit is made of, `gate_counts()` says what it
  costs, and both are worth knowing.
- **`with qc.adjoint():`** runs a block backwards: the buffer is reversed *and* every
  gate is replaced by its inverse, since $(U_k\cdots U_1)^{-1} = U_1^\dagger \cdots
  U_k^\dagger$. It works for any block because every gate is unitary — and unitary means
  invertible means **no information is lost**, in contrast to a classical `AND`, which
  destroys a bit (and, by Landauer, dissipates $k_B T\ln 2$ doing it).
- `H.adjoint()` is `H`, `S.adjoint()` is `S†`, `Rz.adjoint()` negates the angle, and
  `block.adjoint()` inverts a hundred-gate subroutine. One vocabulary, every scale.
- **Measurement has no adjoint** and cannot be controlled, so qsim refuses it inside any
  scope. It is the only non-unitary operation in the library.
- **`with qc.control(c):`** runs a block only where `c` is $|1\rangle$. With `c` in
  superposition the result is a superposition of the block having run and not having run
  — the engine of phase estimation and of Shor's algorithm.
- Controls are **accumulated, not decomposed**: nesting two control scopes equals listing
  two controls, and a multiply-controlled gate is just more sliced axes. Real hardware
  must compile that into one- and two-qubit gates; the slice is the meaning being
  compiled *to*.
- Both scopes work by **record mode** — the one exception to eager execution. Inside a
  scope, ops are buffered, transformed on exit, and only then run. A scope that raises
  discards its buffer, so the state is never left half-transformed.
- **`with qc.ancilla(n) as scratch:`** borrows scratch qubits and verifies on exit that
  they are back in $|0\dots0\rangle$ and unentangled — one probability checks both
  conditions — raising `DirtyAncillaError` otherwise. There is no quantum garbage
  collection: dropping a handle does not release a qubit, because discarding an entangled
  qubit silently turns a pure state into a mixed one.
- A dirty ancilla **destroys interference**, and the demonstration is the heart of this
  notebook: $H$ then $H$ gives $|0\rangle$ with certainty; insert a `CNOT` into a scratch
  qubit and it becomes a coin flip. **Nobody looked at the scratch qubit.** Its mere
  existence as a which-path record is enough, because amplitudes only cancel along paths
  that end in indistinguishable states.
- The effect is **continuous**: a partial record gives partial interference, and the
  entanglement entropy of the system tracks the loss of contrast exactly.
- **Bennett's trick** — compute, copy the answer out, uncompute — is how you use scratch
  space without paying for it. It costs about a factor of two in gates and it is
  non-negotiable; Phase 4's arithmetic is built out of it top to bottom.

- **`with qsim.within(V, q):`** does V, runs the body, and undoes V — the conjugation
  $V U V^\dagger$ that basis changes, oracles and Fourier-space arithmetic are all built
  from. Only V is recorded; the body stays eager and inspectable. Inverting a sandwich
  inverts only its filling, $(VUV^\dagger)^\dagger = V U^\dagger V^\dagger$, and
  controlling one controls every layer, because control distributes over a product.
- **The history is a tape.** `qc.checkpoint()` marks a position and `qc.rewind(mark)`
  runs the inverse of everything since, restoring the state exactly — nothing is saved,
  because unitaries lose nothing. The inverse gates are *appended* to the history: the
  state goes back, the record says how. `rewind` refuses across a measurement (no
  inverse rule exists), across an allocation or ancilla release (the axes would have
  moved), and inside an open scope.
- **`qc.on_op(fn)`** attaches an observer called after every gate *and* every
  measurement, with the state already updated; `handle.remove()` detaches it. Hooks may
  not emit ops. An entropy trace is a hook, not a feature.

## Next: notebook 05

Section 4 built a which-path detector without calling it that. A qubit lost its
coherence to a single scratch qubit that nobody read — which is exactly the experiment
physicists have been running with light and mirrors for a century.

`05-interferometers.ipynb` gives those circuits their physical names. It turns out a
Hadamard *is* a 50/50 beam splitter, so the H-sandwich you have been building since
notebook 01 is a Mach–Zehnder interferometer, and the scratch qubit above is a which-path
detector. With those labels in place the canonical experiments follow directly: fringes
that vanish as a detector learns more, the conservation law $V^2 + D^2 = 1$ that says
interference and which-path knowledge are one resource split two ways, a bomb you can
detect without setting it off, and Feynman's Stern–Gerlach filters.

## And then: notebook 06

Section 4 showed a qubit lose its coherence to a single scratch qubit that nobody read.
Now ask what happens when the scratch qubit is not one you allocated.

A real qubit sits in a laboratory. Photons scatter off it, stray fields nudge it, the
substrate it is printed on jiggles. Each of those interactions is a `CNOT`-like coupling
into some degree of freedom you did not allocate, cannot address, and will never
uncompute — and there are astronomically many of them. **A dirty ancilla is a tiny
environment**, and the environment is a dirty ancilla with $10^{23}$ qubits.

`06-decoherence.ipynb` runs exactly the experiment you have just seen, with the ancilla
relabelled as "the environment" and never traced out — because in qsim the reduced view
is a *choice made by the Inspector*, not something that happens to the state. You will
watch coherence decay as the number of environment qubits grows, see why some states
(the "pointer states") survive while superpositions of them do not, and undo the damage
in a quantum eraser. The conclusion is the one this project keeps circling: the classical
world is not something added on top of the quantum one. It is what the quantum world
looks like from inside, once the records have been written.